# Chapter 18: Semi-Supervised Learning

Semi-supervised learning is a branch of machine learning that combines a small amount of labeled data with a large amount of unlabeled data during training. This is extremely useful because labeling data can be expensive and time-consuming, while unlabeled data is often cheap and abundant.

## 1. Label Propagation and Label Spreading
Graph-based algorithms that model the data points as nodes in a connected graph.
- **Edges**: Weighted by similarity between points.
- **Propagation**: Labels propagate from labeled nodes along the edges to unlabeled nodes.
- **Label Spreading**: A variant that uses a normalized Laplacian matrix and adds a regularization term to make it robust to noise.

## 2. Pseudo-Labeling (Self-Training)
An iterative wrapper-method:
1. Train a classifier on the small set of labeled data.
2. Predict class probabilities for the unlabeled data.
3. Select unlabeled samples whose predicted probability exceeds a high threshold (e.g. $\ge 0.9$).
4. Add these samples to the training set as labeled, using their predicted label ("pseudo-label").
5. Retrain the classifier on the expanded training set and repeat.


## 🛠️ Key Functions to Keep in Mind
* **Graph-based Propagation**:
  * `LabelPropagation(kernel, n_neighbors)`: Propagates labels along a similarity graph.
  * `LabelSpreading(kernel, gamma)`: Label Propagation with noise-regularization on normalized graph Laplacian.
  * `model.transduction_`: Array containing training labels where unlabeled samples (value -1) are filled with predicted labels.
* **Self-Training**:
  * `model.predict_proba(X)`: Predicted class probabilities (essential to filter by pseudo-labeling threshold).



In [ ]:

from sklearn.semi_supervised import LabelPropagation
from sklearn.datasets import make_classification
import numpy as np

# Make data where -1 denotes unlabeled points
X, y = make_classification(n_samples=100, n_features=4, n_classes=2, random_state=42)
y_masked = y.copy()
y_masked[80:] = -1 # Mask 20% labels as unlabeled

lp = LabelPropagation().fit(X, y_masked)
print("Transduced Labels for unlabeled points:", lp.transduction_[80:])




---
## Exercises

Now it is your turn! Run the verification code cell after each exercise to check your answers.

### Exercise 1: Import Label Propagation (Easy)
Import `LabelPropagation` from `sklearn.semi_supervised` and instantiate it. Store it in a variable named `lp_easy`.


In [ ]:
# TODO: Import and instantiate LabelPropagation
lp_easy = ____
print(lp_easy)

In [ ]:
from learntools import ch18
ch18.step_1.check(lp_easy)
# ch18.step_1.hint()
# ch18.step_1.solution()


### Exercise 2: Count Unlabeled Samples (Easy)
In semi-supervised learning, unlabeled samples are represented by a label of `-1`. Given labels `y_masked`, count the number of unlabeled samples. Store it in `unlabeled_count`.


In [ ]:
y_masked = np.array([0, 1, -1, 0, -1, 1, -1])
# TODO: Count how many labels are equal to -1
unlabeled_count = ____
print(unlabeled_count)

In [ ]:
ch18.step_2.check(unlabeled_count)
# ch18.step_2.hint()
# ch18.step_2.solution()


### Exercise 3: Import Label Spreading (Easy)
Import `LabelSpreading` from `sklearn.semi_supervised` and instantiate it. Store it in a variable named `ls_easy`.


In [ ]:
# TODO: Import and instantiate LabelSpreading
ls_easy = ____
print(ls_easy)

In [ ]:
ch18.step_3.check(ls_easy)
# ch18.step_3.hint()
# ch18.step_3.solution()


### Exercise 4: Graph-based Label Propagation (Medium)
Train a `LabelPropagation` model on the dataset where **90%** of labels are masked as unlabeled (`-1`).
1. Use `kernel='knn'` and `n_neighbors=5`.
2. Fit the model on training features `X_train` and labels `y_train` (containing `-1` masks).
3. Compute the accuracy score of the propagated labels on the test mask items. Store your fitted model object in `lp_model` and the accuracy score in `acc`.


In [ ]:

from sklearn.semi_supervised import LabelPropagation
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=6, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Mask 90% of training labels
np.random.seed(42)
mask = np.random.rand(len(y_train)) < 0.9
y_train_masked = y_train.copy()
y_train_masked[mask] = -1

# TODO: Train LabelPropagation model and evaluate accuracy on test set
lp_model = ____
acc = ____

print(f"Transduction Accuracy: {acc:.4f}")




In [ ]:
ch18.step_4.check(lp_model, acc)
# ch18.step_4.hint()
# ch18.step_4.solution()


### Exercise 5: Self-Training Pseudo-Labeling Loop (Medium)
Implement a single iteration of Pseudo-Labeling.
Write a function `pseudo_labeling(X, y_masked, threshold=0.9)` that:
1. Splits the training set into labeled indices (where `y_masked != -1`) and unlabeled indices (where `y_masked == -1`).
2. Trains a `LogisticRegression` classifier on the labeled data.
3. Predicts probabilities for the unlabeled data.
4. Identifies samples where the predicted probability of the predicted class is strictly greater than or equal to `threshold`.
5. Adds these highly confident samples and their predicted labels to the labeled dataset.
6. Retrains a final `LogisticRegression` classifier on the combined dataset and returns this fitted classifier object.


In [ ]:

from sklearn.linear_model import LogisticRegression

def pseudo_labeling(X, y_masked, threshold=0.9):
    # TODO: Implement pseudo-labeling iteration
    pass

# Test on a small set
X_mock = np.random.randn(10, 2)
y_mock = np.array([0, 1, 0, 1, 0, -1, -1, -1, -1, -1])
clf = pseudo_labeling(X_mock, y_mock)
print("Classifier trained successfully:", clf)




In [ ]:
ch18.step_5.check(pseudo_labeling)
# ch18.step_5.hint()
# ch18.step_5.solution()


### Exercise 6: Label Spreading Kernel Hyperparameter Tuning (Hard)
Evaluate the impact of the Radial Basis Function (RBF) kernel `gamma` parameter on the `LabelSpreading` model.
The concentric moons dataset is non-linear.
Evaluate `gamma` values `[0.1, 1.0, 10.0, 100.0]`.
1. For each gamma, fit `LabelSpreading(kernel='rbf', gamma=gamma)` on the training set `(X_train, y_train_masked)`.
2. Compute the accuracy of the propagated labels against the actual labels of the test mask indices (where `y_train_masked == -1`).
3. Store the best gamma found in `best_gamma` and its corresponding accuracy in `best_acc`.


In [ ]:

from sklearn.semi_supervised import LabelSpreading
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=500, noise=0.15, random_state=42)
y_masked = y.copy()
np.random.seed(42)
mask = np.random.rand(len(y)) < 0.8
y_masked[mask] = -1 # Mask 80% labels

# TODO: Loop over gammas, train LabelSpreading, find best_gamma and best_acc
best_gamma = 0.0
best_acc = 0.0

print(f"Best Gamma: {best_gamma} with Test Mask Accuracy: {best_acc:.4f}")




In [ ]:
ch18.step_6.check(best_gamma, best_acc)
# ch18.step_6.hint()
# ch18.step_6.solution()
